In [1]:
# This notebook reads in the discretised input data and then preprocesses the model features
# Firstly, values deemed excessively high/low are capped
# Relevant binary features and normally/log-normally features are standardised accordingly
# Training and test sets are split - 70% train, 10% validation, 20% test
# Resulting datasets are saved to file.

In [2]:
%matplotlib inline
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pandas import DataFrame

In [3]:
disc_inp_data = pd.read_csv("/home/yusuf/Yusuf/Kuliah/AI/Project/ICU/JC-Reproduce/data/processed/discretised_input_data.csv")

In [4]:
# add rewards - sparsely for now; reward function shaping comes in a separate script
disc_inp_data['reward'] = 0
for i in disc_inp_data.index:
    if i == 0:
        continue
    else:
        if disc_inp_data.loc[i, 'icustayid'] != disc_inp_data.loc[i-1, 'icustayid']:
            if disc_inp_data.loc[i-1, 'died_in_hosp'] == 1:
                disc_inp_data.loc[i-1,'reward'] = -100
            elif disc_inp_data.loc[i-1, 'died_in_hosp'] == 0:
                disc_inp_data.loc[i-1,'reward'] = 100
            else:
                print "error in row", i-1
if disc_inp_data.loc[len(disc_inp_data)-1, 'died_in_hosp'] == 1:
    disc_inp_data.loc[len(disc_inp_data)-1, 'reward'] = -100
elif disc_inp_data.loc[len(disc_inp_data)-1, 'died_in_hosp'] == 0:
     disc_inp_data.loc[len(disc_inp_data)-1, 'reward'] = 100
print disc_inp_data['reward'].value_counts()

 0      257667
 100     18063
-100      2885
Name: reward, dtype: int64


In [5]:
# now split into train/validation/test sets
import random
unique_ids = disc_inp_data['icustayid'].unique()
random.shuffle(unique_ids)
train_sample = 0.7
val_sample = 0.1
test_sample = 0.2
train_num = int(len(unique_ids) * 0.7)
val_num = int(len(unique_ids)*0.1) + train_num
train_ids = unique_ids[:train_num]
val_ids = unique_ids[train_num:val_num]
test_ids = unique_ids[val_num:]

In [6]:
train_set = DataFrame()
train_set = disc_inp_data.loc[disc_inp_data['icustayid'].isin(train_ids)]

val_set = DataFrame()
val_set = disc_inp_data.loc[disc_inp_data['icustayid'].isin(val_ids)]

test_set = DataFrame()
test_set = disc_inp_data.loc[disc_inp_data['icustayid'].isin(test_ids)]

In [7]:
binary_fields = ['gender','mechvent','re_admission']
norm_fields= ['age','Weight_kg','GCS','HR','SysBP','MeanBP','DiaBP','RR','Temp_C','FiO2_1',
    'Potassium','Sodium','Chloride','Glucose','Magnesium','Calcium',
    'Hb','WBC_count','Platelets_count','PTT','PT','Arterial_pH','paO2','paCO2',
    'Arterial_BE','HCO3','Arterial_lactate','SOFA','SIRS','Shock_Index',
    'PaO2_FiO2','cumulated_balance', 'elixhauser', 'Albumin', u'CO2_mEqL', 'Ionised_Ca']
log_fields = ['max_dose_vaso','SpO2','BUN','Creatinine','SGOT','SGPT','Total_bili','INR',
              'input_total','input_4hourly','output_total','output_4hourly', 'bloc']

In [8]:
# normalise binary fields
train_set[binary_fields] = train_set[binary_fields] - 0.5 
val_set[binary_fields] = val_set[binary_fields] - 0.5 
test_set[binary_fields] = test_set[binary_fields] - 0.5 

/home/yusuf/Yusuf/Kuliah/AI/Project/ICU/JC-Reproduce/preproc_venv/lib/python2.7/site-packages/pandas/core/frame.py:3391: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: http://pandas.pydata.org/pandas-docs/stable/indexing.html#indexing-view-versus-copy
  self[k1] = value[k2]


In [9]:
# normal distn fields
for item in norm_fields:
    av = train_set[item].mean()
    std = train_set[item].std()
    train_set[item] = (train_set[item] - av) / std
    val_set[item] = (val_set[item] - av) / std
    test_set[item] = (test_set[item] - av) / std

/home/yusuf/Yusuf/Kuliah/AI/Project/ICU/JC-Reproduce/preproc_venv/lib/python2.7/site-packages/ipykernel_launcher.py:5: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: http://pandas.pydata.org/pandas-docs/stable/indexing.html#indexing-view-versus-copy
  """
/home/yusuf/Yusuf/Kuliah/AI/Project/ICU/JC-Reproduce/preproc_venv/lib/python2.7/site-packages/ipykernel_launcher.py:6: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: http://pandas.pydata.org/pandas-docs/stable/indexing.html#indexing-view-versus-copy
  
/home/yusuf/Yusuf/Kuliah/AI/Project/ICU/JC-Reproduce/preproc_venv/lib/python2.7/site-packages/ipykernel_launcher.py:7: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.


In [10]:
# log normal fields
train_set[log_fields] = np.log(0.1 + train_set[log_fields])
val_set[log_fields] = np.log(0.1 + val_set[log_fields])
test_set[log_fields] = np.log(0.1 + test_set[log_fields])
for item in log_fields:
    av = train_set[item].mean()
    std = train_set[item].std()
    train_set[item] = (train_set[item] - av) / std
    val_set[item] = (val_set[item] - av) / std
    test_set[item] = (test_set[item] - av) / std

/home/yusuf/Yusuf/Kuliah/AI/Project/ICU/JC-Reproduce/preproc_venv/lib/python2.7/site-packages/ipykernel_launcher.py:8: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: http://pandas.pydata.org/pandas-docs/stable/indexing.html#indexing-view-versus-copy
  
/home/yusuf/Yusuf/Kuliah/AI/Project/ICU/JC-Reproduce/preproc_venv/lib/python2.7/site-packages/ipykernel_launcher.py:9: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: http://pandas.pydata.org/pandas-docs/stable/indexing.html#indexing-view-versus-copy
  if __name__ == '__main__':
/home/yusuf/Yusuf/Kuliah/AI/Project/ICU/JC-Reproduce/preproc_venv/lib/python2.7/site-packages/ipykernel_launcher.py:10: SettingWithCopyWarning: 
A value is trying to be set on a copy of a 

In [11]:
train_set.head()

,bloc,icustayid,charttime,gender,age,elixhauser,re_admission,died_in_hosp,died_within_48h_of_out_time,mortality_90d,...,input_total,input_4hourly,output_total,output_4hourly,cumulated_balance,SOFA,SIRS,vaso_input,iv_input,reward
7,-2.289867,11,6898241400,0.5,1.175508,0.922189,0.5,0,0,0,...,-3.287212,-1.50302,-2.580138,-1.877165,-0.135102,1.613511,-1.557590,0.0,0.0,0
8,-1.462783,11,6898255800,0.5,1.175508,0.922189,0.5,0,0,0,...,-3.287212,-1.50302,-0.182592,0.663949,-0.171250,1.044761,-1.557590,0.0,0.0,0
9,-0.964629,11,6898270200,0.5,1.175508,0.922189,0.5,0,0,0,...,-3.287212,-1.50302,0.043747,0.723205,-0.215258,1.044761,-0.596961,0.0,0.0,0
10,-0.607019,11,6898284600,0.5,1.175508,0.922189,0.5,0,0,0,...,-3.287212,-1.50302,0.171716,0.733776,-0.260837,1.329136,-0.596961,0.0,0.0,0
11,-0.327856,11,6898299000,0.5,1.175508,0.922189,0.5,0,0,0,...,-3.287212,-1.50302,0.237974,0.636546,-0.293842,1.329136,-0.596961,0.0,0.0,0


In [12]:
train_set.to_csv('/home/yusuf/Yusuf/Kuliah/AI/Project/ICU/JC-Reproduce/data/processed/rl_train_set_unscaled.csv',index = False)
val_set.to_csv('/home/yusuf/Yusuf/Kuliah/AI/Project/ICU/JC-Reproduce/data/processed/rl_val_set_unscaled.csv', index = False)
test_set.to_csv('/home/yusuf/Yusuf/Kuliah/AI/Project/ICU/JC-Reproduce/data/processed/rl_test_set_unscaled.csv', index = False)

In [13]:
# scale features to [0,1] in train set, similar in val and test
import copy
scalable_fields = copy.deepcopy(binary_fields)
scalable_fields.extend(norm_fields)
scalable_fields.extend(log_fields)
for col in scalable_fields:
    minimum = min(train_set[col])
    maximum = max(train_set[col])
    train_set[col] = (train_set[col] - minimum)/(maximum-minimum)
    val_set[col] = (val_set[col] - minimum)/(maximum-minimum)
    test_set[col] = (test_set[col] - minimum)/(maximum-minimum)

/home/yusuf/Yusuf/Kuliah/AI/Project/ICU/JC-Reproduce/preproc_venv/lib/python2.7/site-packages/ipykernel_launcher.py:9: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: http://pandas.pydata.org/pandas-docs/stable/indexing.html#indexing-view-versus-copy
  if __name__ == '__main__':
/home/yusuf/Yusuf/Kuliah/AI/Project/ICU/JC-Reproduce/preproc_venv/lib/python2.7/site-packages/ipykernel_launcher.py:10: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: http://pandas.pydata.org/pandas-docs/stable/indexing.html#indexing-view-versus-copy
  # Remove the CWD from sys.path while we load stuff.
/home/yusuf/Yusuf/Kuliah/AI/Project/ICU/JC-Reproduce/preproc_venv/lib/python2.7/site-packages/ipykernel_launcher.py:11: SettingWithCopyW

In [14]:
train_set.head()

,bloc,icustayid,charttime,gender,age,elixhauser,re_admission,died_in_hosp,died_within_48h_of_out_time,mortality_90d,...,input_total,input_4hourly,output_total,output_4hourly,cumulated_balance,SOFA,SIRS,vaso_input,iv_input,reward
7,0.000000,11,6898241400,1.0,0.902277,0.428571,1.0,0,0,0,...,0.0,0.0,0.000000,0.000000,0.385647,0.521739,0.00,0.0,0.0,0
8,0.222560,11,6898255800,1.0,0.902277,0.428571,1.0,0,0,0,...,0.0,0.0,0.561900,0.707254,0.385106,0.434783,0.00,0.0,0.0,0
9,0.356608,11,6898270200,1.0,0.902277,0.428571,1.0,0,0,0,...,0.0,0.0,0.614947,0.723746,0.384447,0.434783,0.25,0.0,0.0,0
10,0.452837,11,6898284600,1.0,0.902277,0.428571,1.0,0,0,0,...,0.0,0.0,0.644938,0.726688,0.383765,0.478261,0.25,0.0,0.0,0
11,0.527957,11,6898299000,1.0,0.902277,0.428571,1.0,0,0,0,...,0.0,0.0,0.660466,0.699627,0.383271,0.478261,0.25,0.0,0.0,0


In [15]:
train_set.to_csv('/home/yusuf/Yusuf/Kuliah/AI/Project/ICU/JC-Reproduce/data/processed/rl_train_set_scaled.csv',index = False)
val_set.to_csv('/home/yusuf/Yusuf/Kuliah/AI/Project/ICU/JC-Reproduce/data/processed/rl_val_set_scaled.csv', index = False)
test_set.to_csv('/home/yusuf/Yusuf/Kuliah/AI/Project/ICU/JC-Reproduce/data/processed/rl_test_set_scaled.csv', index = False)